In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

df = spark.table("retail_project.gold.demand_forecast_features")

feature_cols = ["day_of_week", "month", "lag_1_sales", "lag_7_sales", "rolling_avg_7"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df).select("features", "total_sales")

train, test = data.randomSplit([0.8, 0.2], seed=42)

with mlflow.start_run(run_name="linear_regression_demand_forecast"):
    lr = LinearRegression(featuresCol="features", labelCol="total_sales")
    model = lr.fit(train)

    predictions = model.transform(test)
    evaluator_rmse = RegressionEvaluator(labelCol="total_sales", metricName="rmse")
    evaluator_r2 = RegressionEvaluator(labelCol="total_sales", metricName="r2")

    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)

    mlflow.log_param("features", feature_cols)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.spark.log_model(model, "linear_regression_model", dfs_tmpdir="/Volumes/retail_project/gold/mlflow_tmp")

    print(f"RMSE: {rmse:.2f}, R2: {r2:.3f}")